# Liber · 실제 사진으로 검출기 미세조정 (Colab)

직접 찍은 사진 21장을 **Roboflow에서 라벨**(SAM 3 자동 라벨 초안 → 사람이 검토·수정)한 뒤, 일부는 **학습**에, 나머지는 **시험(채점)**에 쓴다.

| 지난 미세조정(합성 위주)이 실패한 이유 | 이번 대응 |
|---|---|
| 배경에 인터넷의 건물·창문 사진을 써서 **빌딩 사진이 들어간 표지**(공업수학 하)를 책이 아니라고 배움 | 배경 = **직접 찍은 학습용 사진 속 책 없는 영역** |
| 붙여 넣은 책(남의 책장 책등)이 실제 장면과 달라 **세운 책**에 효과 없음 | 책 = **직접 찍은 학습용 사진 속 실제 책**을 오려 씀 (+ Kaggle 표지·책등 보조) |
| 채점이 공개·합성 데이터 기준이라 실제 사진 결과와 달랐음 | **라벨된 실제 시험 사진**으로 권수·재현율 채점 (지금 검출기와 비교) |

- 학습용 실제 사진은 학습 목록에 `REPEAT`배 반복 + 학습 중 증강(좌우 반전 · 회전 · 밝기/색조 · 크기)
- 시험용 사진(`TEST_STEMS`)의 책·배경은 학습·합성 어디에도 쓰지 않음
- 결과: `liber_web_models_real.zip` (웹앱용 `e_unified.onnx` · `e_unified.json`) — **시험 사진에서 지금 검출기보다 나을 때만** 웹앱에 적용

### 실행 전
- Drive `내 드라이브/liber/`: `books_obb_built.zip` · `det_obb_best.pt` · (선택) `archive.zip`
- 🔑 보안 비밀 `ROBOFLOW_API_KEY` (노트북 액세스 켜기) · Roboflow 프로젝트 `liber-books`에 **버전(Version)을 하나 만들어 둠**
- 런타임 **A100 GPU** → **모두 실행** (약 1~1.5시간)

In [ ]:
# Cell 0 · 패키지
!pip -q install ultralytics==8.4.183 roboflow==1.7.1 onnx onnxruntime onnxslim
!apt-get -qq install -y fonts-nanum > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 1 · 준비 · 설정
import json
import os
import random
import re
import shutil
import subprocess
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from google.colab import drive, files, userdata
from matplotlib import font_manager
from roboflow import Roboflow
from ultralytics import YOLO

drive.mount("/content/drive")
for f in Path("/usr/share/fonts/truetype/nanum").glob("*.ttf"):
    font_manager.fontManager.addfont(str(f))
plt.rcParams["font.family"] = "NanumGothic"

# ---- 설정
RF_PROJECT = "liber-books"     # Roboflow 프로젝트 이름 (워크스페이스는 API 키의 기본 워크스페이스)
RF_VERSION = None              # None = 최신 버전
TEST_STEMS = ["IMG_1530", "IMG_1593", "IMG_1587"]   # 시험용(학습에 안 씀): 책장 · 책상 위 세운 책 · 평평한 표지
TEST_EXTRA = 0.2               # 나머지 사진 중 이 비율을 시험용에 더 (무작위, 고정 시드)
REPEAT = 20                    # 학습용 실제 사진 반복 배수
N_SYN, N_BG_ONLY = 1500, 200   # 실제 책·배경으로 만드는 합성 사진 수 · 책 없는 배경 사진 수
FT = dict(epochs=30, patience=10, imgsz=960, batch=32, workers=8, cache="disk",
          optimizer="SGD", lr0=0.002, lrf=0.1, warmup_epochs=1,
          degrees=15.0, fliplr=0.5, flipud=0.0, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
          mosaic=1.0, close_mosaic=10, save_period=-1)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DEVICE = 0
TIMES = {}
DRIVE = Path("/content/drive/MyDrive/liber")
WORK = Path("/content/work_real")
DS = WORK / "books_obb"
REAL = WORK / "real"           # Roboflow에서 받은 실제 사진 (학습용 · 시험용)
WORK.mkdir(parents=True, exist_ok=True)
BASE_PT = DRIVE / "det_obb_best.pt"
RUN = DRIVE / "runs" / "det_obb_real"
OUT_PT = DRIVE / "det_obb_best_real.pt"
IMGSZ = FT["imgsz"]
for need in (DRIVE / "books_obb_built.zip", BASE_PT):
    assert need.exists(), f"{need} 없음 → 본학습 노트북을 먼저 실행하세요"
if not DS.exists():
    subprocess.run(["unzip", "-q", "-o", str(DRIVE / "books_obb_built.zip"), "-d", str(WORK)], check=True)
cfg = yaml.safe_load(open(DS / "data.yaml"))
cfg["path"] = str(DS)
yaml.safe_dump(cfg, open(DS / "data.yaml", "w"), sort_keys=False)
print({s: len(list((DS / "images" / s).iterdir())) for s in ("train", "val")})

In [ ]:
# Cell 2 · 공통 함수
def to_obb(values, w, h):
    """YOLO 라벨 한 줄(클래스 제외) → 정규화 4꼭짓점. 다각형(분할 라벨)은 최소 외접 회전 사각형으로"""
    if len(values) == 4:
        cx, cy, bw, bh = values
        pts = np.array([[cx - bw / 2, cy - bh / 2], [cx + bw / 2, cy - bh / 2], [cx + bw / 2, cy + bh / 2], [cx - bw / 2, cy + bh / 2]])
    elif len(values) >= 6 and len(values) % 2 == 0:
        poly = (np.array(values).reshape(-1, 2) * [w, h]).astype(np.float32)
        pts = cv2.boxPoints(cv2.minAreaRect(poly)) / [w, h]
    else:
        return None
    pts = np.clip(pts, 0, 1)
    return pts.reshape(-1).tolist() if cv2.contourArea(pts.astype(np.float32)) > 1e-5 else None


def write_obb_label(path, obbs):
    path.write_text("".join("0 " + " ".join(f"{v:.6f}" for v in o) + "\n" for o in obbs))


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in Path(lbl_path).read_text().splitlines() if len(l.split()) >= 9]


def rect_frame(poly):
    box = cv2.boxPoints(cv2.minAreaRect(np.asarray(poly, np.float32)))
    for i in range(4):
        for step in (1, -1):
            p0, p1, p3 = box[i], box[(i + step) % 4], box[(i - step) % 4]
            u, v = p1 - p0, p3 - p0
            if np.linalg.norm(u) >= np.linalg.norm(v) and v[0] * u[1] - v[1] * u[0] > 0:
                return p0, u, v
    return box[0], box[1] - box[0], box[3] - box[0]


def crop_rotated(img, poly):
    p0, u, v = rect_frame(poly)
    L, S = max(int(round(np.linalg.norm(u))), 1), max(int(round(np.linalg.norm(v))), 1)
    M = cv2.getAffineTransform(np.float32([p0, p0 + u, p0 + v]), np.float32([[0, 0], [0, L], [S, 0]]))
    return cv2.warpAffine(img, M, (S, L), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def limit_side(img, m):
    s = m / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def poly_iou(a, b):
    inter, _ = cv2.intersectConvexConvex(a, b)
    union = cv2.contourArea(a) + cv2.contourArea(b) - inter
    return inter / union if union > 0 else 0.0


def stem_of(p):
    """Roboflow 파일 이름 'IMG_1530_jpg.rf.<해시>.jpg' → 'IMG_1530'"""
    return re.sub(r"_(jpe?g|png|heic|webp)$", "", Path(p).stem.split(".rf.")[0], flags=re.I)

In [ ]:
# Cell 3 · [실제 사진] Roboflow 라벨 받기 → 회전 상자 → 학습용 / 시험용 나누기 · ✅ 라벨 확인
t0 = time.time()
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
proj = rf.workspace().project(RF_PROJECT)
ver = RF_VERSION or max(int(str(v.version).split("/")[-1]) for v in proj.versions())
proj.version(ver).download("yolov8", location=str(WORK / "rf_real"), overwrite=True)

items = []
for img_path in sorted((WORK / "rf_real").rglob("*")):
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"} or img_path.parent.name != "images":
        continue
    lbl = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    obbs = [o for l in (lbl.read_text().splitlines() if lbl.exists() else []) if len(l.split()) >= 5
            for o in [to_obb([float(v) for v in l.split()[1:]], w, h)] if o]
    items.append({"stem": stem_of(img_path), "path": img_path, "obbs": obbs})
items = list({it["stem"]: it for it in items}.values())   # 같은 사진이 여러 분할에 있으면 하나만

rest = sorted(it["stem"] for it in items if it["stem"] not in TEST_STEMS)
random.Random(SEED).shuffle(rest)
test_set = set(TEST_STEMS) | set(rest[:int(len(rest) * TEST_EXTRA)])
for split in ("train", "test"):
    for d in ("images", "labels"):
        (REAL / split / d).mkdir(parents=True, exist_ok=True)
for it in items:
    it["split"] = "test" if it["stem"] in test_set else "train"
    dst = REAL / it["split"] / "images" / f"{it['stem']}.jpg"
    shutil.copy2(it["path"], dst)
    write_obb_label(REAL / it["split"] / "labels" / f"{it['stem']}.txt", it["obbs"])
summary = pd.DataFrame([{"사진": it["stem"], "분할": it["split"], "책(라벨)": len(it["obbs"])} for it in items]).sort_values(["분할", "사진"])
display(summary)
missing = [s for s in TEST_STEMS if s not in {it["stem"] for it in items}]
if missing:
    print("⚠️ Roboflow에 없는 시험 사진:", missing)
TIMES["라벨 받기"] = (time.time() - t0) / 60

fig, axes = plt.subplots((len(items) + 4) // 5, 5, figsize=(20, 4.5 * ((len(items) + 4) // 5)), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, it in zip(axes.flat, sorted(items, key=lambda x: (x["split"], x["stem"]))):
    img = cv2.cvtColor(cv2.imread(str(REAL / it["split"] / "images" / f"{it['stem']}.jpg")), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for o in it["obbs"]:
        cv2.polylines(img, [(np.array(o).reshape(4, 2) * [w, h]).astype(np.int32)], True, (255, 0, 0) if it["split"] == "train" else (0, 120, 255), max(3, w // 250))
    ax.imshow(img)
    ax.set_title(f"{it['stem']} · {it['split']} · {len(it['obbs'])}권", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 · [재료] 학습용 실제 사진에서: 책 오리기 + 책 없는 배경 영역 잘라내기 (+ Kaggle 표지 · 기존 책등 보조)
t0 = time.time()
rng = random.Random(SEED)
real_books, backgrounds = [], []
for p in sorted((REAL / "train" / "images").glob("*.jpg")):
    img = cv2.imread(str(p))
    h, w = img.shape[:2]
    polys = read_obb(REAL / "train" / "labels" / f"{p.stem}.txt", w, h)
    for q in polys:
        c = crop_rotated(img, q)
        if min(c.shape[:2]) >= 40:
            real_books.append(limit_side(c, 1000))
    occ = np.zeros((h, w), np.uint8)   # 책이 있는 곳
    cv2.fillPoly(occ, [q.astype(np.int32) for q in polys], 1)
    for _ in range(60):                # 책이 거의 없는(3% 미만) 사각 영역 → 배경
        ch, cw = int(h * rng.uniform(0.35, 0.8)), int(w * rng.uniform(0.35, 0.8))
        y, x = rng.randint(0, h - ch), rng.randint(0, w - cw)
        if occ[y:y + ch, x:x + cw].mean() < 0.03:
            backgrounds.append(limit_side(img[y:y + ch, x:x + cw], 1280))
spines, covers = [], []
train_imgs = [p for p in sorted((DS / "images" / "train").iterdir()) if p.stem.startswith("rf_") and "_rep" not in p.stem]
rng.shuffle(train_imgs)
for p in train_imgs[:1500]:
    img = cv2.imread(str(p))
    for q in read_obb(DS / "labels" / "train" / f"{p.stem}.txt", img.shape[1], img.shape[0])[:2]:
        c = crop_rotated(img, q)
        if c.shape[0] >= 120 and c.shape[0] >= 2.5 * c.shape[1]:
            spines.append(limit_side(c, 900))
cov_zip = next((z for z in (DRIVE / "archive.zip", DRIVE / "book_covers.zip") if z.exists()), None)
if cov_zip:
    cdir = WORK / "covers"
    if not cdir.exists():
        subprocess.run(["unzip", "-q", "-o", str(cov_zip), "book-covers/*", "-d", str(cdir)], check=True)
    cps = sorted(cdir.rglob("*.jpg"))
    rng.shuffle(cps)
    covers = [c for c in (cv2.imread(str(p)) for p in cps[:1500]) if c is not None and min(c.shape[:2]) >= 100]
TIMES["재료"] = (time.time() - t0) / 60
print(f"실제 책 {len(real_books)} · 배경 {len(backgrounds)} · 책등 {len(spines)} · 표지 {len(covers)}")
assert real_books and backgrounds, "학습용 실제 사진에서 책·배경을 얻지 못했습니다 (라벨 확인)"

In [ ]:
# Cell 5 · [합성] 실제 배경 위에 책 1~3권 (실제 책 60% · 표지 20% · 책등 20%) — 세움 · 누움 · 쌓음 · 평평한 표지(90° 포함)
#   + 책 없는 배경만 있는 사진(빈 라벨) → '빈 배경은 책이 아님'
def rotate_obj(obj, ang):
    h, w = obj.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
    c, s = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * s + w * c) + 2, int(h * c + w * s) + 2
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return (cv2.warpAffine(obj, M, (nw, nh)), cv2.warpAffine(np.full((h, w), 255, np.uint8), M, (nw, nh)),
            cv2.transform(np.float32([[[0, 0], [w, 0], [w, h], [0, h]]]), M)[0])


def paste(canvas, rot, mask, x0, y0, r):
    H, W = canvas.shape[:2]
    nh, nw = mask.shape
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + nw), min(H, y0 + nh)
    if xb <= xa or yb <= ya:
        return
    m = mask[ya - y0:yb - y0, xa - x0:xb - x0].astype(np.float32) / 255.0
    o = rot[ya - y0:yb - y0, xa - x0:xb - x0].astype(np.float32) * r.uniform(0.8, 1.1)
    sh = cv2.GaussianBlur(m, (0, 0), max(2.0, 0.04 * min(nw, nh)))
    dx, dy = int(0.03 * nw) + 1, int(0.02 * nh) + 1
    sh = np.pad(sh, ((dy, 0), (dx, 0)))[:m.shape[0], :m.shape[1]]
    region = canvas[ya:yb, xa:xb].astype(np.float32) * (1 - 0.3 * sh[..., None])
    canvas[ya:yb, xa:xb] = np.clip(region * (1 - m[..., None]) + o * m[..., None], 0, 255).astype(np.uint8)


def pick_obj(r):
    u = r.random()
    if u < 0.6 or not (covers or spines):
        o = r.choice(real_books)
        return o, ("cover" if o.shape[0] < 2 * o.shape[1] else "spine")
    if u < 0.8 and covers:
        return r.choice(covers), "cover"
    return r.choice(spines or real_books), "spine"


def make_one(r):
    canvas = r.choice(backgrounds).copy()
    H, W = canvas.shape[:2]
    frame = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
    placed, desk, stack = [], r.uniform(0.6, 0.95) * H, None
    for _ in range(r.choice([1, 1, 2, 2, 3])):
        for _try in range(10):
            obj, kind = pick_obj(r)
            obj = obj.copy()
            if kind == "spine" and r.random() < 0.4:
                obj = cv2.resize(obj, (int(obj.shape[1] * r.uniform(1.3, 2.5)), obj.shape[0]))
            pose = (("flat" if r.random() < 0.6 else "stand") if kind == "cover" else r.choices(["stand", "lie"], [0.6, 0.4])[0])
            if pose == "stand":
                target, ang = r.uniform(0.3, 0.85) * H, r.uniform(-8, 8)
            elif pose == "lie":
                target, ang = r.uniform(0.3, 0.8) * W, 90 + r.uniform(-6, 6)
            else:
                u = r.random()
                ang = r.uniform(-25, 25) if u < 0.5 else (r.choice([90, -90]) + r.uniform(-10, 10)) if u < 0.75 else r.uniform(0, 360)
                target = r.uniform(0.4, 0.85) * min(H, W)
            s = target / obj.shape[0]
            obj = cv2.resize(obj, (max(8, int(obj.shape[1] * s)), max(8, int(obj.shape[0] * s))), interpolation=cv2.INTER_AREA)
            rot, mask, corners = rotate_obj(obj, ang)
            nh, nw = mask.shape
            if pose == "stand":
                cx, cy = r.uniform(0.1, 0.9) * W, desk - nh / 2
            elif pose == "lie" and stack is not None and r.random() < 0.6:
                cx, cy = stack[0] + r.uniform(-0.05, 0.05) * W, stack[1] - nh / 2
            elif pose == "flat":
                cx, cy = r.uniform(0.3, 0.7) * W, r.uniform(0.3, 0.7) * H
            else:
                cx, cy = r.uniform(0.15, 0.85) * W, r.uniform(0.45, 0.92) * H
            x0, y0 = int(cx - nw / 2), int(cy - nh / 2)
            poly = (corners + np.float32([x0, y0])).astype(np.float32)
            area = cv2.contourArea(poly)
            vis, _ = cv2.intersectConvexConvex(poly, frame)
            if area <= 0 or vis / area < 0.85 or any(cv2.intersectConvexConvex(poly, q)[0] / min(area, cv2.contourArea(q)) > 0.15 for q in placed):
                continue
            paste(canvas, rot, mask, x0, y0, r)
            placed.append(poly)
            if pose == "lie":
                stack = (cx, poly[:, 1].min())
            break
    if not placed:
        return None
    canvas = np.clip(canvas.astype(np.float32) * r.uniform(0.85, 1.1) + r.uniform(-12, 12), 0, 255).astype(np.uint8)
    return canvas, [(p / np.float32([W, H])).clip(0, 1).reshape(-1).tolist() for p in placed]


t0 = time.time()
n = Counter()
for i in range(N_SYN):
    out = make_one(random.Random(SEED * 1000 + i))
    if out:
        cv2.imwrite(str(DS / "images" / "train" / f"syn3__{i:05d}.jpg"), out[0], [cv2.IMWRITE_JPEG_QUALITY, 90])
        write_obb_label(DS / "labels" / "train" / f"syn3__{i:05d}.txt", out[1])
        n["합성"] += 1
for i in range(N_BG_ONLY):
    r = random.Random(SEED * 3000 + i)
    bg = r.choice(backgrounds)
    bg = cv2.flip(bg, 1) if r.random() < 0.5 else bg
    cv2.imwrite(str(DS / "images" / "train" / f"bg3__{i:05d}.jpg"), bg, [cv2.IMWRITE_JPEG_QUALITY, 90])
    (DS / "labels" / "train" / f"bg3__{i:05d}.txt").write_text("")
    n["배경만"] += 1
# 학습용 실제 사진 REPEAT배 (학습 중 증강이 매번 다르게 적용됨)
for p in sorted((REAL / "train" / "images").glob("*.jpg")):
    for k in range(REPEAT):
        dst = DS / "images" / "train" / f"real__{p.stem}__{k:02d}.jpg"
        try:
            os.link(p, dst)
        except OSError:
            shutil.copy2(p, dst)
        shutil.copy2(REAL / "train" / "labels" / f"{p.stem}.txt", DS / "labels" / "train" / f"real__{p.stem}__{k:02d}.txt")
        n["실제(반복)"] += 1
for c in DS.rglob("*.cache"):
    c.unlink()
TIMES["합성·추가"] = (time.time() - t0) / 60
print(dict(n))

ps = sorted((DS / "images" / "train").glob("syn3__*.jpg"))
random.Random(1).shuffle(ps)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p in zip(axes.flat, ps[:8]):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for q in read_obb(DS / "labels" / "train" / f"{p.stem}.txt", w, h):
        cv2.polylines(img, [q.astype(np.int32)], True, (255, 0, 0), max(2, w // 300))
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("✅ 합성 사진 (실제 배경 · 실제 책)")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 · [미세조정] 본학습 검출기에서 이어서 — 세션이 끊기면 Cell 0~5를 다시 실행한 뒤 이 셀 (last.pt에서 이어서)
t0 = time.time()
last = RUN / "weights" / "last.pt"
if last.exists():
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:
        print("이어서 할 학습 없음:", e)
else:
    YOLO(str(BASE_PT)).train(data=str(DS / "data.yaml"), project=str(RUN.parent), name=RUN.name, exist_ok=True,
                             device=DEVICE, seed=SEED, **FT)
shutil.copy2(RUN / "weights" / "best.pt", OUT_PT)
TIMES["미세조정"] = (time.time() - t0) / 60
print(f"→ {OUT_PT} · {TIMES['미세조정']:.0f}분")

In [ ]:
# Cell 7 · ✅ 실제 시험 사진 채점: 지금 검출기(본학습) ↔ 미세조정 — 사진별 권수 · 재현율 · 정밀도, 신뢰도 기준별
models = {"지금(본학습)": YOLO(str(BASE_PT)), "미세조정(실제)": YOLO(str(OUT_PT))}
tests = sorted((REAL / "test" / "images").glob("*.jpg"))
GRID = [round(x, 2) for x in np.arange(0.2, 0.61, 0.05)]
rows, preds = [], {}
for k, m in models.items():
    for p in tests:
        img = cv2.imread(str(p))
        gt = read_obb(REAL / "test" / "labels" / f"{p.stem}.txt", img.shape[1], img.shape[0])
        r = m.predict(img, imgsz=IMGSZ, conf=min(GRID), device=DEVICE, verbose=False)[0]
        P = [q.astype(np.float32) for q in r.obb.xyxyxyxy.cpu().numpy()] if r.obb is not None else []
        C = r.obb.conf.cpu().numpy() if r.obb is not None else np.array([])
        preds[(k, p.stem)] = (P, C)
        for c in GRID:
            pp = [q for q, s in zip(P, C) if s >= c]
            used, tp = set(), 0
            for g in gt:
                cand = [(poly_iou(g, q), j) for j, q in enumerate(pp) if j not in used]
                if cand and max(cand)[0] >= 0.5:
                    used.add(max(cand)[1])
                    tp += 1
            rows.append({"모델": k, "conf": c, "사진": p.stem, "실제": len(gt), "검출": len(pp), "맞음": tp})
ev = pd.DataFrame(rows)
by_conf = ev.groupby(["모델", "conf"]).apply(lambda g: pd.Series({
    "권수 정확(사진)": f"{(g['실제'] == g['검출']).sum()}/{len(g)}",
    "|권수 오차| 합": int((g["실제"] - g["검출"]).abs().sum()),
    "재현율(%)": round(g["맞음"].sum() / max(g["실제"].sum(), 1) * 100, 1),
    "정밀도(%)": round(g["맞음"].sum() / max(g["검출"].sum(), 1) * 100, 1)}))
display(by_conf.unstack(0))
WEB_CONF = float(ev[ev["모델"] == "미세조정(실제)"].assign(err=lambda d: (d["실제"] - d["검출"]).abs())
                 .groupby("conf")["err"].sum().idxmin())
BASE_CONF = 0.45   # 지금 웹앱 설정
pick = lambda k, c: ev[(ev["모델"] == k) & (ev["conf"] == c)].set_index("사진")[["실제", "검출", "맞음"]]
side = pick("지금(본학습)", BASE_CONF).join(pick("미세조정(실제)", WEB_CONF), lsuffix=" · 지금", rsuffix=" · 미세조정")
print(f"사진별 (지금 conf {BASE_CONF} · 미세조정 conf {WEB_CONF})")
display(side)
err_base = int((side["실제 · 지금"] - side["검출 · 지금"]).abs().sum())
err_ft = int((side["실제 · 미세조정"] - side["검출 · 미세조정"]).abs().sum())
BETTER = err_ft < err_base
print(f"|권수 오차| 합: 지금 {err_base} → 미세조정 {err_ft} → {'✅ 나아짐 (웹앱 적용 권장)' if BETTER else '❌ 나아지지 않음 (적용하지 않음 권장)'}")
ev.to_csv(DRIVE / "eval_real_test.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Cell 8 · ✅ 시험 사진 그림: [지금 | 미세조정] 상자 (각자 고른 신뢰도 기준 이상)
fig, axes = plt.subplots(len(tests), 2, figsize=(14, 6 * len(tests)), squeeze=False)
for row, p in zip(axes, tests):
    img = cv2.imread(str(p))
    for ax, (k, c) in zip(row, [("지금(본학습)", BASE_CONF), ("미세조정(실제)", WEB_CONF)]):
        P, C = preds[(k, p.stem)]
        vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t = max(3, img.shape[1] // 250)
        n_keep = 0
        for q, s in zip(P, C):
            if s >= c:
                cv2.polylines(vis, [q.astype(np.int32)], True, (255, 0, 0), t)
                n_keep += 1
        ax.imshow(vis)
        ax.set_title(f"[{p.stem}] {k} · {n_keep}권 (실제 {len(read_obb(REAL / 'test' / 'labels' / f'{p.stem}.txt', img.shape[1], img.shape[0]))}권)", fontsize=12)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 · 웹앱용 모델 → liber_web_models_real.zip (Drive 저장 + 자동 다운로드) — 적용 여부는 Cell 7 결과로 판단
import onnxruntime as ort
WEB = WORK / "liber_web_models_real"
if WEB.exists():
    shutil.rmtree(WEB)
WEB.mkdir()
onnx_path = YOLO(str(OUT_PT), task="obb").export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)
shutil.copy2(onnx_path, WEB / "e_unified.onnx")
(WEB / "e_unified.json").write_text(json.dumps({"imgsz": IMGSZ, "conf": WEB_CONF, "zoom": 0}))
s = ort.InferenceSession(str(WEB / "e_unified.onnx"), providers=["CPUExecutionProvider"])
o = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, IMGSZ, IMGSZ), np.float32)})[0]
assert o.shape[1] == 6, o.shape
(WEB / "RESULT.txt").write_text(f"시험 사진 |권수 오차| 합: 지금 {err_base} → 미세조정 {err_ft} · 신뢰도 {WEB_CONF} · {'나아짐' if BETTER else '나아지지 않음'}\n")
z = shutil.make_archive(str(WORK / "liber_web_models_real"), "zip", WEB)
shutil.copy2(z, DRIVE / "liber_web_models_real.zip")
display(pd.Series(TIMES, name="분").round(1).to_frame())
print((WEB / "RESULT.txt").read_text())
files.download(z)